# VASP Preprocessing

Notebook to create the VASP input files (POTCAR, )

In [2]:
import os
os.chdir("/Users/adrianaladera/Desktop/MIT/RESEARCH/Github_repos/SolidState-dont-Evaporate/")

from utils.VaspPreprocess import VaspPreprocess
from utils.packages import *

# https://my.nersc.gov/queuewaittimes.php
hours = 21 # change based on what the heatmap says lol
nedos = 4000
density = 3000

## Making relax directory based on the POSCAR in the main directory

Resulting tree will look like this:

    - original/path/to/POSCAR
    |
    |-- scf
        |- POSCAR
        |- POTCAR
        |- KPOINTS
        |- INCAR
    |-- relax
        |- POSCAR
        |- POTCAR
        |- KPOINTS
        |- INCAR


If you want to relax your structure, execute the following cell.

In [2]:
# path to where your POSCAR is stored; don't include the name of the POSCAR
root = "/Users/adrianaladera/Desktop/MIT/RESEARCH/VASP_calculations/funky-form_ML/Si_lattices/"
filename = "POSCAR_0.vasp"
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
cunt = 0
for mocha in os.listdir(root):

    # mocha = "3amino/refined/"

    if "uniform" in mocha or "biaxial" in mocha:
        filename = "POSCAR"
        path = f"{root}/{mocha}/"
        print(filename)

        # creating directories
        if not os.path.exists(f"{path}/relax"):
            os.mkdir(f"{path}/relax")

        # renames filename to POSCAR for VASP input
        os.system(f"cp {path}/{filename} {path}/relax/POSCAR")
        print(f"{filename} copied to relax/POSCAR")

        structure = Structure.from_file(f"{path}/relax/POSCAR")
        structure.get_space_group_info = SpacegroupAnalyzer(structure, symprec=0.5, angle_tolerance=10.0).get_space_group_info

        VaspPreprocess.run_cpu(mocha, f"{path}/relax/", hours=hours)
        VaspPreprocess.kpoints_by_density(f"{path}/relax/", structure, density=1000)
        VaspPreprocess.make_potcar(f"{path}/relax/")
        VaspPreprocess.relax_incar(f"{path}/relax", isif=2, nsw=100, ibrion=2, isym=2, ivdw=11, lcharg=False, lwave=False) # checks to see if POTCAR exists, then creates INCAR if True
    cunt += 1

POSCAR
POSCAR copied to relax/POSCAR


AttributeError: 'SpacegroupAnalyzer' object has no attribute 'get_space_group_info'

In [ ]:
root = "/Users/adrianaladera/Desktop/MIT/RESEARCH/Github_repos/periodic-hamiltonian/data"
from ase.io import read, write
import os

frames = read(f"{root}/C2.xyz", index=":")
for i, f in enumerate(frames):
    if not os.path.exists(f"{root}/C2_frame_{i}"):
        os.mkdir(f"{root}/C2_frame_{i}")
    write(f"{root}/C2_frame_{i}/frame_{i}.vasp", f)
    # write()

## Make a self-consistent field (SCF) folder either from a relaxed structure or an initial structure

IMPORTANT: must choose the correct file for which you want to run an SCF calculation. Be sure that if you want to run an SCF on a relaxed structure, you are copying the final CONTCAR from the relax/ folder into the SCF folder!

In [3]:
from pymatgen.io.vasp.inputs import Kpoints
import math

def kpoints_by_density_strained(path, structure, density=1000):
    lattice = structure.lattice
    lengths = lattice.abc
    ngrid = density / structure.num_sites
    mult = (ngrid * structure.volume) ** (1/3)
    num_div = [max(math.floor(mult / l), 1) for l in lengths]
    kpoints = Kpoints.gamma_automatic(num_div, [0, 0, 0])
    kpoints.write_file(f"{path}/KPOINTS")

root = "/Users/adrianaladera/Desktop/MIT/RESEARCH/VASP_calculations/funky-form_ML/perovskites/0_strain"
isrelaxed = False ######## CHANGE ME ##########
make_scf = False
from pymatgen.io.vasp import Poscar

for structure in os.listdir(root):

    name = structure

    if "3" in structure:
        path = f"{root}/{structure}/"
        # print(path)
        # filename = "Mo-bcc-" + structure + ".vasp"
        # print(f"{path}/{filename}")
        # print(structure)
    #     if not os.path.exists(f"{root}{structure[:-4]}"):
    #         os.mkdir(f"{root}{structure[:-4]}")
    #     path = f"{root}/{structure[:-4]}/"
        filename = "POSCAR"

        if make_scf:
            tag = "scf"
            if not os.path.exists(f"{path}/{tag}"):
                os.mkdir(f"{path}/{tag}")
        else:
            tag = ''

        if isrelaxed:
            os.system(f"cp {path}/relax/CONTCAR {path}/{tag}/POSCAR") ########## CHANGE ME #######
            os.system(f"cp {path}/relax/run.slurm {path}/{tag}/run.slurm")
            print(f"relax/CONTCAR copied to scf/POSCAR")
        else:
            if os.path.exists(f"{path}/{filename}"):
                struct = Structure.from_file(f"{path}/{filename}")
                ass = Poscar(struct)
                ass.write_file(f"{path}/{tag}/POSCAR")
                # os.system(f"cp {path}/{filename} {path}/scf/POSCAR")
                print(f"{filename} copied to {tag}/POSCAR")

        if os.path.exists(f"{path}/{tag}/POSCAR"):
            struct = Structure.from_file(f"{path}/{tag}/POSCAR")
            # VaspPreprocess.run_cpu(str(name), f"{path}/{tag}/", hours=hours)
            kpoints_by_density_strained(f"{path}/{tag}/", struct, density=density)
            # VaspPreprocess.kpoints_by_density(f"{path}/{tag}/", struct, density=density)
            # make_potcar(f"{path}/scf/")
            VaspPreprocess.make_potcar(f"{path}/{tag}/")
            # scf_incar(f"{path}/scf/", isym=2, lwave=False, lcharg=True)
            # import os
            # import re
            # command = f"grep \"ZVAL\" {path}/POTCAR"
            # os.system(command)
            # output = os.popen(command).read();
            # energy_vals = [float(i) for i in output.replace(";", "").split() if re.match(r'^-?\d+(?:\.\d+)$', i) is not None]
            # print(energy_vals)
            VaspPreprocess.scf_incar(f"{path}/{tag}/", isym=2, nbands=True, lwave=True, lcharg=True)#, ispin=2, magmom=2.2) # checks to see if POTCAR exists, then creates INCAR if True
            

POSCAR copied to /POSCAR
cat /Users/adrianaladera/Desktop/MIT/research/POTCARS_PBE.54/K_sv/POTCAR /Users/adrianaladera/Desktop/MIT/research/POTCARS_PBE.54/Nb_sv/POTCAR /Users/adrianaladera/Desktop/MIT/research/POTCARS_PBE.54/O/POTCAR > /Users/adrianaladera/Desktop/MIT/RESEARCH/VASP_calculations/funky-form_ML/perovskites/0_strain/KNbO3////POTCAR
POTCAR written
   ENMAX  =  259.264; ENMIN  =  194.448 eV
   ENMAX  =  293.235; ENMIN  =  219.927 eV
   ENMAX  =  400.000; ENMIN  =  300.000 eV
1 1 3
direct
   POMASS =   39.098; ZVAL   =    9.000    mass and valenz
   POMASS =   92.910; ZVAL   =   13.000    mass and valenz
   POMASS =   16.000; ZVAL   =    6.000    mass and valenz
[ 9. 13.  6.] [1. 1. 3.] 120
Self-consistent field (SCF) INCAR written
POSCAR copied to /POSCAR
cat /Users/adrianaladera/Desktop/MIT/research/POTCARS_PBE.54/O/POTCAR /Users/adrianaladera/Desktop/MIT/research/POTCARS_PBE.54/Sr_sv/POTCAR /Users/adrianaladera/Desktop/MIT/research/POTCARS_PBE.54/Ti/POTCAR > /Users/adriana

In [ ]:

path = "/Users/adrianaladera/Desktop/yourmom/"
# filename = "POSCAR"
isrelaxed = False ######## CHANGE ME ##########
from pymatgen.io.vasp import Poscar
from pymatgen.core import Molecule

fuckyou = "water_rotated_cell_test0.vasp"
os.system(f"cp {root}/{fuckyou} {root}/POSCAR")
# structure = Molecule.from_file(f"{root}/{fuckyou}")

# file = Poscar(structure)
# file.write(f"{path}POSCAR")

VaspPreprocess.run_cpu(str(name), f"{path}", hours=hours)
VaspPreprocess.kpoints_by_density(f"{path}", struct, density=density)# make_potcar(f"{path}/scf/")
VaspPreprocess.make_potcar(f"{path}")
                    # scf_incar(f"{path}/scf/", isym=2, lwave=False, lcharg=True)
VaspPreprocess.scf_incar(f"{path}", isym=2, lwave=False, lcharg=True)

## Electronic Structure preprocessing

Resulting tree will look like this:

    - original/path/to/POSCAR
    |
    |-- relax
    |-- scf
    |-- dos
        |- POSCAR
        |- POTCAR
        |- KPOINTS
        |- INCAR
        |- CHGCAR
    |-- band
        |- POSCAR
        |- POTCAR
        |- KPOINTS
        |- INCAR
        
The default functional is PBE in VASP. To use hybrid functionals (in this example, B3LYP), additional parameters have been added to the dos_incar() and band_incar() functions. B3LYP can be used simply by setting `ishybrid=True`.

In [ ]:


ishybrid = False
root = "/Users/adrianaladera/Desktop/MIT/RESEARCH/VASP_calculations/"
filename = "ligand_FINAL.vasp"
for structure in os.listdir(root):
    name = structure
    path = f"{root}/{structure}"
    if "CuS" in structure:
        print(path)
        potcar = f"{path}/scf/POTCAR"
        struct = Structure.from_file(f"{path}/scf/POSCAR")
        ### PARCHG
        # VaspPreprocess.make_parchg_incar(path)
        # VaspPreprocess.kpoints_by_density(f"{path}/parchg/", struct, density=density)
        # VaspPreprocess.run_cpu(str(name), f"{path}/parchg/")
        # VaspPreprocess.make_potcar(f"{path}/parchg/")

        # if not os.path.exists(f"{path}/scf_pchg"):
        #     os.mkdir(f"{path}/scf_pchg")
        # os.system(f"cp {path}/scf/POSCAR {path}/scf_pchg")
        # VaspPreprocess.make_potcar(f"{path}/scf_pchg/")
        # VaspPreprocess.scf_incar(f"{path}/scf_pchg", isym=-1, lwave=True, lcharg=False)
        # VaspPreprocess.kpoints_by_density(f"{path}/scf_pchg/", struct, density=density)
        # VaspPreprocess.run_cpu(str(name), f"{path}/scf_pchg/", hours=hours)

        VaspPreprocess.band_kpoints(struct, path, kpath_type='s') # creates the band/ and scf_band/ directories and primitive cells
        VaspPreprocess.make_potcar(f"{path}/scf_band/")
        VaspPreprocess.scf_incar(f"{path}/scf_band", isym=2, lwave=False, lcharg=True)
        struct = Structure.from_file(f"{path}/scf_band/POSCAR")
        VaspPreprocess.kpoints_by_density(f"{path}/scf_band/", struct, density=density)
        VaspPreprocess.run_cpu(str(name), f"{path}/scf_band/", hours=hours)

        VaspPreprocess.band_incar(f"{path}/scf_band/POTCAR", f"{path}/band", ishybrid=ishybrid, lwave=False, lcharg=True)
        VaspPreprocess.make_potcar(f"{path}/band/")
        VaspPreprocess.run_cpu(str(name), f"{path}/band/", hours=hours)

        # if "structure" not in root or "inorganic" not in root:
        #     VaspPreprocess.dos_incar(f"{path}/scf/POTCAR", f"{path}/", nedos=nedos, ishybrid=ishybrid, lwave=False, lcharg=True) # creates dos/ directory
        #     VaspPreprocess.run_cpu(str(name), f"{path}/dos/", hours=hours)
        # else:
        #     os.system(f"rm -r {path}/scf")

In [ ]:
'/from pymatgen.core import Element

for s in struct:
    # print(s.specie)
    elem = Element(s.specie)
    print(s.specie, elem.block)
    # print(type(elem.block))
    if elem.block == 'd':
        x = 8
    print(x, s.specie)

In [ ]:
from pymatgen.core import Element, Structure

# Load structure
# struct = Structure.from_file("blah blah mocha file name")

# dictionary mapping orbital block → your chosen x value
block_x = {
    's': 2,
    'p': 4,
    'd': 8,
    'f': 14,
}

# assume zval_dict maps element symbols to ZVALs parsed from POTCAR
# e.g., {"Ag": 11.0, "S": 6.0, ...}
# zval_dict = get_zvals_from_potcar("POTCAR")
zval_dict = {"Ag": 11.0, "S": 6.0, "C": 5, "H": 2, "O":6}

# compute the total
total = sum(
    (zval_dict[s.specie.symbol] / 2) + block_x.get(Element(s.specie).block, 0)
    for s in struct
)
total

In [ ]:
ass = 3.10284
round(ass)

In [ ]:
ass = [128] * 7
ass